In [1]:
"""SEED-IV 4-Class Emotion Recognition: Full STMAE + MSC-TimesNet.

=============================================================
Architecture:
  - Spatio-Temporal MAE: Joint temporal frame masking (mt) and spatial region
masking (ms).
  - Temporal Transformer Encoder + Scalp Topology Spatial Transformer Encoder.
  - Symmetrical Decoders optimizing L_rec = L_t + L_s.
  - Downstream Backbone: Multi-frequency MSC-TimesNet (FFT Periodicity + 2D
Inception Convolutions).

Evaluation Protocols:
  1. Leave-One-Subject-Out (LOSO) Cross-Validation (15 folds).
  2. Round-Robin Cross-Session Evaluation:
       - Train Session 1 -> Test Session 2 & Test Session 3 separately ->
       Average.
       - Train Session 2 -> Test Session 1 & Test Session 3 separately ->
       Average.
       - Train Session 3 -> Test Session 1 & Test Session 2 separately ->
       Average.
"""

'SEED-IV 4-Class Emotion Recognition: Full STMAE + MSC-TimesNet.\n\n=============================================================\nArchitecture:\n  - Spatio-Temporal MAE: Joint temporal frame masking (mt) and spatial region\nmasking (ms).\n  - Temporal Transformer Encoder + Scalp Topology Spatial Transformer Encoder.\n  - Symmetrical Decoders optimizing L_rec = L_t + L_s.\n  - Downstream Backbone: Multi-frequency MSC-TimesNet (FFT Periodicity + 2D\nInception Convolutions).\n\nEvaluation Protocols:\n  1. Leave-One-Subject-Out (LOSO) Cross-Validation (15 folds).\n  2. Round-Robin Cross-Session Evaluation:\n       - Train Session 1 -> Test Session 2 & Test Session 3 separately ->\n       Average.\n       - Train Session 2 -> Test Session 1 & Test Session 3 separately ->\n       Average.\n       - Train Session 3 -> Test Session 1 & Test Session 2 separately ->\n       Average.\n'

In [2]:
import copy
from collections import defaultdict
import math
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import scipy.io as sio
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# =====================================================================
# 1. SEED-IV CONFIGURATION & PATHS
# =====================================================================
DATA_SEARCH_PATHS = [
    "/kaggle/input/datasets/phhasian0710/seed-iv/eeg_feature_smooth",
    "/kaggle/input/seed-iv/eeg_feature_smooth",
    "./eeg_feature_smooth",
    ".",
]
OUTPUT_DIR = "/kaggle/working/seediv_stmae_timesnet"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_CHANNELS = 62
NUM_CLASSES = 4  # SEED-IV: 0: Neutral, 1: Sad, 2: Fear, 3: Happy
RAW_DIM = 10  # 5 log-PSD + 5 DE
TRIALS_PER_SESSION = 24
WINDOW_LENGTH = 10  # T=10 frames (40s sequence per window)
STRIDE = 1

TRIAL_LABELS_BY_SESSION = {
    1: [1, 2, 3, 0, 2, 0, 0, 1, 0, 1, 2, 1, 1, 1, 2, 3, 2, 2, 3, 3, 0, 3, 0, 3],
    2: [2, 1, 3, 0, 0, 2, 0, 2, 3, 3, 2, 3, 2, 0, 1, 1, 2, 1, 0, 3, 0, 1, 3, 1],
    3: [1, 2, 2, 1, 3, 3, 3, 1, 1, 2, 1, 0, 2, 3, 3, 0, 2, 3, 0, 0, 2, 0, 1, 0],
}

CHANNEL_NAMES = [
    "FP1", "FPZ", "FP2", "AF3", "AF4", "F7", "F5", "F3", "F1", "FZ",
    "F2", "F4", "F6", "F8", "FT7", "FC5", "FC3", "FC1", "FCZ", "FC2",
    "FC4", "FC6", "FT8", "T7", "C5", "C3", "C1", "CZ", "C2", "C4",
    "C6", "T8", "TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6",
    "TP8", "P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8",
    "PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8", "CB1", "O1", "OZ",
    "O2", "CB2",
]
assert len(CHANNEL_NAMES) == NUM_CHANNELS

# ---- Stage A: Full Spatio-Temporal STMAE ----
D_MODEL = 64
TEMPORAL_HEADS = 4
TEMPORAL_LAYERS = 2
SPATIAL_HEADS = 4
SPATIAL_LAYERS = 2
TEMPORAL_MASK_RATIO = 0.30  # Frame-level mask rate mt
SPATIAL_MASK_RATIO = 0.40   # Channel-level mask rate ms
REGION_MASK_CHANCE = 0.60
STMAE_EPOCHS = 35
STMAE_LR = 1e-3
STMAE_BATCH_SIZE = 128

# ---- Stage B: MSC-TimesNet Backbone ----
CLASSIFIER_HIDDEN_SIZE = 128
CLASSIFIER_BLOCKS = 2
TOP_FREQUENCIES = 3
MAX_EPOCHS = 50
MIN_EPOCHS = 15
PATIENCE_EPOCHS = 10
LEARNING_RATE = 1e-3
BATCH_SIZE = 128
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP = 1.0
LABEL_SMOOTHING = 0.05
VALIDATION_FRACTION = 0.2
MIXUP_STRENGTH = 0.2
CHANNEL_DROPOUT_RATE = 0.1

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed=RANDOM_SEED):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)


# =====================================================================
# 2. 2D SCALP TOPOLOGY & REGIONAL PARTITIONING
# =====================================================================
def build_scalp_coords():
  rows = [
      (["FP1", "FPZ", "FP2"], 0.95),
      (["AF3", "AF4"], 0.80),
      (["F7", "F5", "F3", "F1", "FZ", "F2", "F4", "F6", "F8"], 0.62),
      (["FT7", "FC5", "FC3", "FC1", "FCZ", "FC2", "FC4", "FC6", "FT8"], 0.42),
      (["T7", "C5", "C3", "C1", "CZ", "C2", "C4", "C6", "T8"], 0.20),
      (["TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6", "TP8"], -0.02),
      (["P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8"], -0.25),
      (["PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8"], -0.50),
      (["CB1", "O1", "OZ", "O2", "CB2"], -0.72),
  ]
  coords = {}
  for names, y in rows:
    n = len(names)
    xs = np.linspace(-1.0, 1.0, n) if n > 1 else np.array([0.0])
    for name, x in zip(names, xs):
      coords[name] = (float(x), float(y))
  return np.array([coords[c] for c in CHANNEL_NAMES], dtype=np.float32)


def build_regions():
  regions = defaultdict(list)
  coords = build_scalp_coords()
  for i, _ in enumerate(CHANNEL_NAMES):
    x, y = coords[i]
    if abs(x) > 0.6 and -0.10 < y < 0.50:
      key = "temporal_left" if x < 0 else "temporal_right"
    elif y >= 0.55:
      key = "frontal"
    elif y >= 0.10:
      key = "central"
    elif y >= -0.35:
      key = "parietal"
    else:
      key = "occipital"
    regions[key].append(i)
  return {k: np.array(v, dtype=np.int64) for k, v in regions.items()}


SCALP_COORDS = build_scalp_coords()
REGIONS = build_regions()


class ScalpPositionalEncoding(nn.Module):

  def __init__(self, coords, d_model):
    super().__init__()
    self.register_buffer("coords", torch.tensor(coords, dtype=torch.float32))
    self.mlp = nn.Sequential(
        nn.Linear(2, d_model), nn.GELU(), nn.Linear(d_model, d_model)
    )

  def forward(self, x):
    return x + self.mlp(self.coords)


class TemporalPositionalEncoding(nn.Module):

  def __init__(self, max_len, d_model):
    super().__init__()
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32)
        * -(math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    self.register_buffer("pe", pe.unsqueeze(0))

  def forward(self, x):
    return x + self.pe[:, : x.shape[1], :]


# =====================================================================
# 3. SPATIO-TEMPORAL MASKED AUTOENCODER (STMAE)
# =====================================================================
class SpatioTemporalMAE(nn.Module):

  def __init__(
      self,
      raw_dim=RAW_DIM,
      d_model=D_MODEL,
      t_heads=TEMPORAL_HEADS,
      t_layers=TEMPORAL_LAYERS,
      s_heads=SPATIAL_HEADS,
      s_layers=SPATIAL_LAYERS,
      seq_len=WINDOW_LENGTH,
  ):
    super().__init__()
    self.d_model = d_model
    self.seq_len = seq_len

    self.feat_proj = nn.Linear(raw_dim, d_model)
    self.temporal_pos = TemporalPositionalEncoding(seq_len, d_model)
    self.spatial_pos = ScalpPositionalEncoding(SCALP_COORDS, d_model)

    self.temporal_mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
    self.spatial_mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
    nn.init.normal_(self.temporal_mask_token, std=0.02)
    nn.init.normal_(self.spatial_mask_token, std=0.02)

    t_layer = nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=t_heads,
        dim_feedforward=d_model * 2,
        dropout=0.1,
        batch_first=True,
        norm_first=True,
        activation="gelu",
    )
    self.temporal_encoder = nn.TransformerEncoder(t_layer, num_layers=t_layers)

    s_layer = nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=s_heads,
        dim_feedforward=d_model * 2,
        dropout=0.1,
        batch_first=True,
        norm_first=True,
        activation="gelu",
    )
    self.spatial_encoder = nn.TransformerEncoder(s_layer, num_layers=s_layers)

    self.spatial_decoder = nn.TransformerEncoder(s_layer, num_layers=s_layers)
    self.temporal_decoder = nn.TransformerEncoder(t_layer, num_layers=t_layers)
    self.recon_head = nn.Sequential(
        nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, raw_dim)
    )

  def apply_masks(self, x, t_mask, s_mask):
    x_masked = torch.where(
        t_mask.unsqueeze(-1).unsqueeze(-1).expand_as(x),
        self.temporal_mask_token.expand_as(x),
        x,
    )
    x_masked = torch.where(
        s_mask.unsqueeze(1).unsqueeze(-1).expand_as(x_masked),
        self.spatial_mask_token.expand_as(x_masked),
        x_masked,
    )
    return x_masked

  def encode(self, x, t_mask=None, s_mask=None):
    B, T, C, _ = x.shape
    h = self.feat_proj(x)
    if t_mask is not None and s_mask is not None:
      h = self.apply_masks(h, t_mask, s_mask)

    # Frame-Level Temporal Attention across T
    h = h.permute(0, 2, 1, 3).reshape(B * C, T, self.d_model)
    h = self.temporal_pos(h)
    h_temp = self.temporal_encoder(h)
    h_temp = h_temp.reshape(B, C, T, self.d_model).permute(0, 2, 1, 3)

    # Topological Spatial Attention across C
    h_spat = h_temp.reshape(B * T, C, self.d_model)
    h_spat = self.spatial_pos(h_spat)
    z_lat = self.spatial_encoder(h_spat)
    return z_lat.reshape(B, T, C, self.d_model)

  def forward(self, x, t_mask, s_mask):
    B, T, C, _ = x.shape
    z = self.encode(x, t_mask=t_mask, s_mask=s_mask)

    # Dual Symmetrical Decoders: Spatial -> Temporal
    z_s = self.spatial_decoder(z.reshape(B * T, C, self.d_model)).reshape(
        B, T, C, self.d_model
    )
    z_t = self.temporal_decoder(
        z_s.permute(0, 2, 1, 3).reshape(B * C, T, self.d_model)
    )
    z_out = z_t.reshape(B, C, T, self.d_model).permute(0, 2, 1, 3)

    recon = self.recon_head(z_out)
    joint_mask = (
        (t_mask.unsqueeze(-1).unsqueeze(-1) | s_mask.unsqueeze(1).unsqueeze(-1))
        .expand_as(x)
        .bool()
    )
    loss = F.mse_loss(recon[joint_mask], x[joint_mask])
    return recon, loss


def sample_spatiotemporal_mask(batch_size, device=DEVICE):
  n_t_mask = max(1, int(TEMPORAL_MASK_RATIO * WINDOW_LENGTH))
  t_mask = torch.zeros(
      batch_size, WINDOW_LENGTH, dtype=torch.bool, device=device
  )
  for b in range(batch_size):
    t_mask[b, torch.randperm(WINDOW_LENGTH, device=device)[:n_t_mask]] = True

  s_mask = torch.zeros(
      batch_size, NUM_CHANNELS, dtype=torch.bool, device=device
  )
  region_keys = list(REGIONS.keys())
  for b in range(batch_size):
    if random.random() < REGION_MASK_CHANCE:
      k = random.choice([1, 2])
      for key in random.sample(region_keys, k):
        s_mask[b, torch.tensor(REGIONS[key], device=device)] = True
    else:
      n_s = max(1, int(SPATIAL_MASK_RATIO * NUM_CHANNELS))
      s_mask[b, torch.randperm(NUM_CHANNELS, device=device)[:n_s]] = True
  return t_mask, s_mask


def pretrain_stmae_sequences(X_pt, seq_idx_pt, epochs=STMAE_EPOCHS):
  print("  [+] Starting Spatio-Temporal Masked Autoencoder Pretraining...")
  model = SpatioTemporalMAE().to(DEVICE)
  opt = torch.optim.AdamW(
      model.parameters(), lr=STMAE_LR, weight_decay=1e-4
  )
  sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

  n_windows = len(seq_idx_pt)
  model.train()
  for ep in range(1, epochs + 1):
    perm = torch.randperm(n_windows, device=DEVICE)
    tot_loss, nb = 0.0, 0
    for i in range(0, n_windows, STMAE_BATCH_SIZE):
      b_idx = perm[i : i + STMAE_BATCH_SIZE]
      xb = X_pt[seq_idx_pt[b_idx]]
      t_mask, s_mask = sample_spatiotemporal_mask(xb.shape[0], DEVICE)
      _, loss = model(xb, t_mask, s_mask)

      opt.zero_grad()
      loss.backward()
      nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
      opt.step()
      tot_loss += loss.item()
      nb += 1
    sched.step()
    if ep == 1 or ep % 5 == 0:
      print(
          "    -> STMAE Epoch %02d | Joint Recon MSE: %.5f"
          % (ep, tot_loss / max(nb, 1))
      )
  return model


# =====================================================================
# 4. MSC-TIMESNET TEMPORAL BACKBONE & CLASSIFIER
# =====================================================================
def fft_topk_periods(x, k=TOP_FREQUENCIES):
  B, T, d = x.shape
  xf = torch.fft.rfft(x, dim=1)
  amp = xf.abs().mean(dim=2)
  amp[:, 0] = 0.0
  k = min(k, max(amp.shape[1] - 1, 1))
  _, idx = torch.topk(amp, k, dim=1)
  freqs = idx.float().mean(dim=0).round().long().clamp(min=1)
  periods = [max(int(T // f.item()), 1) for f in freqs]
  weights = torch.stack([amp[:, i] for i in freqs], dim=1)
  return periods, F.softmax(weights, dim=1)


class MultiScaleConvBlock(nn.Module):

  def __init__(self, d_model):
    super().__init__()
    h = max(d_model // 4, 8)
    self.b1 = nn.Sequential(
        nn.Conv2d(d_model, h, 1), nn.BatchNorm2d(h), nn.GELU()
    )
    self.b3 = nn.Sequential(
        nn.Conv2d(d_model, h, 3, padding=1), nn.BatchNorm2d(h), nn.GELU()
    )
    self.b5 = nn.Sequential(
        nn.Conv2d(d_model, h, 5, padding=2), nn.BatchNorm2d(h), nn.GELU()
    )
    self.bp = nn.Sequential(
        nn.AvgPool2d(3, stride=1, padding=1),
        nn.Conv2d(d_model, h, 1),
        nn.BatchNorm2d(h),
        nn.GELU(),
    )
    self.fuse = nn.Sequential(
        nn.Conv2d(4 * h, d_model, 1), nn.BatchNorm2d(d_model)
    )

  def forward(self, x):
    return self.fuse(
        torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)
    )


class TimesBlock(nn.Module):

  def __init__(self, d_model, topk=TOP_FREQUENCIES):
    super().__init__()
    self.topk = topk
    self.conv = MultiScaleConvBlock(d_model)
    self.norm = nn.LayerNorm(d_model)

  def forward(self, x):
    B, T, d = x.shape
    periods, weights = fft_topk_periods(x, self.topk)
    outs = []
    for p in periods:
      pad = (math.ceil(T / p) * p) - T
      xp = F.pad(x, (0, 0, 0, pad)) if pad > 0 else x
      Tp = xp.shape[1]
      num_p = Tp // p
      z = xp.permute(0, 2, 1).reshape(B, d, num_p, p)
      z = self.conv(z)
      z = z.reshape(B, d, Tp).permute(0, 2, 1)[:, :T, :]
      outs.append(z)
    agg = (torch.stack(outs, dim=-1) * weights.unsqueeze(1).unsqueeze(1)).sum(
        dim=-1
    )
    return self.norm(agg + x)


class FullModel(nn.Module):

  def __init__(
      self,
      stmae,
      num_classes=NUM_CLASSES,
      hidden_dim=CLASSIFIER_HIDDEN_SIZE,
      finetune_encoder=False,
  ):
    super().__init__()
    self.stmae = stmae
    self.finetune_encoder = finetune_encoder

    in_dim = NUM_CHANNELS * D_MODEL
    self.inp_proj = nn.Sequential(
        nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim)
    )
    self.blocks = nn.ModuleList(
        [TimesBlock(hidden_dim) for _ in range(CLASSIFIER_BLOCKS)]
    )
    self.head = nn.Sequential(
        nn.LayerNorm(hidden_dim),
        nn.Dropout(0.3),
        nn.Linear(hidden_dim, num_classes),
    )

  def forward(self, x):
    B, T, C, _ = x.shape
    if self.finetune_encoder and self.training:
      z = self.stmae.encode(x)
    else:
      with torch.no_grad():
        z = self.stmae.encode(x)

    h = self.inp_proj(z.reshape(B, T, C * D_MODEL))
    for blk in self.blocks:
      h = blk(h)
    return self.head(h.mean(dim=1))


# =====================================================================
# 5. DATA INGESTION FOR SEED-IV
# =====================================================================
def find_seediv_dir():
  for p in DATA_SEARCH_PATHS:
    if os.path.exists(os.path.join(p, "1")):
      return p
  return None


def load_seed_iv_dataset():
  data_dir = find_seediv_dir()
  if data_dir is not None:
    print(f"  [+] Ingesting SEED-IV features from: {data_dir}")
    xs, ys, subs, sess, tris = [], [], [], [], []
    for session in sorted(TRIAL_LABELS_BY_SESSION.keys()):
      sdir = os.path.join(data_dir, str(session))
      if not os.path.isdir(sdir):
        continue
      labels = TRIAL_LABELS_BY_SESSION[session]
      for fname in sorted(os.listdir(sdir)):
        if not fname.endswith(".mat"):
          continue
        subject = int(fname.split("_")[0])
        mat = sio.loadmat(os.path.join(sdir, fname))
        for t in range(1, TRIALS_PER_SESSION + 1):
          dk, pk = f"de_LDS{t}", f"psd_LDS{t}"
          if dk not in mat or pk not in mat:
            continue
          de = np.transpose(
              np.asarray(mat[dk], dtype=np.float32), (1, 0, 2)
          )
          psd = np.transpose(
              np.asarray(mat[pk], dtype=np.float32), (1, 0, 2)
          )
          psd = np.log(np.maximum(psd, 1e-10))
          feat = np.concatenate([psd, de], axis=2)
          n = feat.shape[0]
          xs.append(feat)
          ys.append(np.full(n, labels[t - 1], dtype=np.int64))
          subs.append(np.full(n, subject, dtype=np.int64))
          sess.append(np.full(n, session, dtype=np.int64))
          tris.append(np.full(n, t, dtype=np.int64))

    if len(xs) > 0:
      x = np.concatenate(xs, axis=0)
      y = np.concatenate(ys, axis=0)
      subs = np.concatenate(subs, axis=0)
      sess = np.concatenate(sess, axis=0)
      tri = np.concatenate(tris, axis=0)
      print(f"  [+] Successfully loaded {x.shape[0]} SEED-IV frames.")
      return x, y, subs, sess, tri

  print("  [!] SEED-IV directory not found. Generating matching fallback cohort")
  print("      (15 Subjects x 3 Sessions x 24 Trials, 4 Classes)...")
  n_samples_per_trial = 35
  total_trials = 15 * 3 * 24
  total_frames = total_trials * n_samples_per_trial

  x = np.random.randn(total_frames, NUM_CHANNELS, RAW_DIM).astype(np.float32)
  subs, sess, tri, y = [], [], [], []
  for sb in range(1, 16):
    for se in range(1, 4):
      labels = TRIAL_LABELS_BY_SESSION[se]
      for tr in range(1, TRIALS_PER_SESSION + 1):
        subs.extend([sb] * n_samples_per_trial)
        sess.extend([se] * n_samples_per_trial)
        tri.extend([tr] * n_samples_per_trial)
        y.extend([labels[tr - 1]] * n_samples_per_trial)

  return (
      x,
      np.array(y, dtype=np.int64),
      np.array(subs, dtype=np.int64),
      np.array(sess, dtype=np.int64),
      np.array(tri, dtype=np.int64),
  )


def normalize_subject_session(x, subs, sess):
  xn = x.copy()
  keys = subs * 100 + sess
  for k in np.unique(keys):
    m = keys == k
    blk = xn[m]
    mu = blk.mean(axis=0, keepdims=True)
    sd = blk.std(axis=0, keepdims=True) + 1e-6
    xn[m] = (blk - mu) / sd
  return xn


def build_sequence_index(
    y, subs, sess, tri, seq_len=WINDOW_LENGTH, stride=STRIDE
):
  keys = subs * 1000000 + sess * 10000 + tri
  seqs, labs, s_sub, s_ses, s_tri = [], [], [], [], []
  order = np.argsort(keys, kind="stable")
  for k in np.unique(keys):
    rows = order[keys[order] == k]
    if rows.shape[0] < seq_len:
      continue
    for start in range(0, rows.shape[0] - seq_len + 1, stride):
      win = rows[start : start + seq_len]
      seqs.append(win)
      labs.append(y[win[0]])
      s_sub.append(subs[win[0]])
      s_ses.append(sess[win[0]])
      s_tri.append(tri[win[0]])
  return (
      np.asarray(seqs, dtype=np.int64),
      np.asarray(labs, dtype=np.int64),
      np.asarray(s_sub, dtype=np.int64),
      np.asarray(s_ses, dtype=np.int64),
      np.asarray(s_tri, dtype=np.int64),
  )


# =====================================================================
# 6. TRAINING & EVALUATION UTILITIES
# =====================================================================
def balanced_weights(y):
  cnt = np.bincount(y, minlength=NUM_CLASSES).astype(np.float32)
  cnt[cnt == 0] = 1.0
  w = cnt.sum() / (NUM_CLASSES * cnt)
  return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def lr_at(ep, base_lr, warmup_epochs=5):
  if ep <= warmup_epochs:
    return base_lr * ep / max(1, warmup_epochs)
  prog = (ep - warmup_epochs) / max(1, MAX_EPOCHS - warmup_epochs)
  return base_lr * 0.5 * (1.0 + math.cos(math.pi * min(prog, 1.0)))


@torch.no_grad()
def predict_probs(model, x_pt, seq_idx_pt, rows_pt, batch_size=BATCH_SIZE):
  model.eval()
  out = []
  n = len(rows_pt)
  for i in range(0, n, batch_size):
    b_rows = rows_pt[i : i + batch_size]
    xb = x_pt[seq_idx_pt[b_rows]]
    out.append(F.softmax(model(xb), dim=1).cpu().numpy())
  return np.concatenate(out, axis=0)


def trial_level_scores(probs, y, sub, ses, tri):
  keys = sub * 1000000 + ses * 10000 + tri
  yt, yp = [], []
  for k in np.unique(keys):
    m = keys == k
    yt.append(np.bincount(y[m]).argmax())
    yp.append(probs[m].mean(axis=0).argmax())
  yt, yp = np.array(yt), np.array(yp)
  return (
      accuracy_score(yt, yp),
      balanced_accuracy_score(yt, yp),
      f1_score(yt, yp, average="macro", zero_division=0),
      f1_score(yt, yp, average="weighted", zero_division=0),
      len(yt),
  )


def fit_model(stmae, x_pt, seq_idx_pt, labels, groups, train_rows, seed=42):
  seed_everything(seed)
  model = FullModel(stmae, finetune_encoder=False).to(DEVICE)

  gss = GroupShuffleSplit(
      n_splits=1, test_size=VALIDATION_FRACTION, random_state=seed
  )
  tr_loc, va_loc = next(
      gss.split(
          np.zeros(len(train_rows)), labels[train_rows], groups[train_rows]
      )
  )

  tr_indices = train_rows[tr_loc]
  va_indices = train_rows[va_loc]

  tr_rows_pt = torch.tensor(tr_indices, dtype=torch.long, device=DEVICE)
  va_rows_pt = torch.tensor(va_indices, dtype=torch.long, device=DEVICE)
  tr_labels_pt = torch.tensor(labels[tr_indices], dtype=torch.long, device=DEVICE)

  crit = nn.CrossEntropyLoss(
      weight=balanced_weights(labels[tr_indices]),
      label_smoothing=LABEL_SMOOTHING,
  )
  opt = torch.optim.AdamW(
      model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
  )

  n_tr = len(tr_rows_pt)
  best_f1, best_state, bad = -1.0, None, 0

  for ep in range(1, MAX_EPOCHS + 1):
    cur_lr = lr_at(ep, LEARNING_RATE)
    for g in opt.param_groups:
      g["lr"] = cur_lr

    model.train()
    perm = torch.randperm(n_tr, device=DEVICE)

    for i in range(0, n_tr, BATCH_SIZE):
      b_idx = perm[i : i + BATCH_SIZE]
      xb = x_pt[seq_idx_pt[tr_rows_pt[b_idx]]]
      yb = tr_labels_pt[b_idx]

      if CHANNEL_DROPOUT_RATE > 0:
        mask_keep = (
            torch.rand(xb.size(0), 1, NUM_CHANNELS, 1, device=DEVICE)
            > CHANNEL_DROPOUT_RATE
        ).float()
        xb = xb * mask_keep

      lam = np.random.beta(MIXUP_STRENGTH, MIXUP_STRENGTH)
      perm_b = torch.randperm(xb.size(0), device=DEVICE)
      xb_mix = lam * xb + (1 - lam) * xb[perm_b]
      logits = model(xb_mix)
      loss = lam * crit(logits, yb) + (1 - lam) * crit(logits, yb[perm_b])

      opt.zero_grad()
      loss.backward()
      nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
      opt.step()

    vp = predict_probs(model, x_pt, seq_idx_pt, va_rows_pt)
    vf1 = f1_score(
        labels[va_indices],
        vp.argmax(1),
        average="macro",
        zero_division=0,
    )
    if vf1 > best_f1:
      best_f1, bad = vf1, 0
      best_state = {
          k: v.detach().cpu().clone() for k, v in model.state_dict().items()
      }
    else:
      bad += 1
      if ep >= MIN_EPOCHS and bad >= PATIENCE_EPOCHS:
        break

  if best_state is not None:
    model.load_state_dict(best_state)
  return model


def evaluate_test_rows(
    model, x_pt, seq_idx_pt, labels, s_sub, s_ses, s_tri, test_rows
):
  te_rows_pt = torch.tensor(test_rows, dtype=torch.long, device=DEVICE)
  probs = predict_probs(model, x_pt, seq_idx_pt, te_rows_pt)
  yt = labels[test_rows]
  pred_win = probs.argmax(1)

  win_acc = accuracy_score(yt, pred_win)
  win_bacc = balanced_accuracy_score(yt, pred_win)
  win_macro_f1 = f1_score(yt, pred_win, average="macro", zero_division=0)
  win_weighted_f1 = f1_score(yt, pred_win, average="weighted", zero_division=0)

  t_acc, t_bacc, t_macro_f1, t_weighted_f1, num_trials = trial_level_scores(
      probs, yt, s_sub[test_rows], s_ses[test_rows], s_tri[test_rows]
  )

  return {
      "win_acc": win_acc,
      "win_bacc": win_bacc,
      "win_macro_f1": win_macro_f1,
      "win_weighted_f1": win_weighted_f1,
      "trial_acc": t_acc,
      "trial_bacc": t_bacc,
      "trial_macro_f1": t_macro_f1,
      "trial_weighted_f1": t_weighted_f1,
      "num_windows": len(test_rows),
      "num_trials": num_trials,
  }


# =====================================================================
# 7. PROTOCOL 1: LEAVE-ONE-SUBJECT-OUT (LOSO)
# =====================================================================
def run_loso_evaluation(stmae, x_pt, seq_idx_pt, packs):
  labels, s_sub, s_ses, s_tri = packs[1:]
  groups = s_sub * 1000000 + s_ses * 10000 + s_tri
  unique_subs = np.unique(s_sub)
  loso_records = []

  print("\n" + "=" * 92)
  print(
      f"PROTOCOL 1: SEED-IV 4-CLASS LOSO CROSS-VALIDATION ({len(unique_subs)} SUBJECTS)"
  )
  print("=" * 92)

  for sb in unique_subs:
    tr_rows = np.where(s_sub != sb)[0]
    te_rows = np.where(s_sub == sb)[0]

    model = fit_model(
        copy.deepcopy(stmae), x_pt, seq_idx_pt, labels, groups, tr_rows
    )
    res = evaluate_test_rows(
        model, x_pt, seq_idx_pt, labels, s_sub, s_ses, s_tri, te_rows
    )
    res["fold"] = f"Subject_{sb:02d}"
    loso_records.append(res)

    print(
        f"  -> {res['fold']:<12} | "
        f"WIN: acc={res['win_acc']:.4f} bacc={res['win_bacc']:.4f} macF1={res['win_macro_f1']:.4f} | "
        f"TRIAL: acc={res['trial_acc']:.4f} bacc={res['trial_bacc']:.4f} macF1={res['trial_macro_f1']:.4f} "
        f"({res['num_windows']}w / {res['num_trials']}t)"
    )

  df_loso = pd.DataFrame(loso_records)
  df_loso.to_csv(os.path.join(OUTPUT_DIR, "results_seediv_loso.csv"), index=False)

  print("\n" + "-" * 92)
  print("LOSO GRAND SUMMARY (Mean over 15 subjects):")
  print("-" * 92)
  metrics = [
      "win_acc", "win_bacc", "win_macro_f1", "win_weighted_f1",
      "trial_acc", "trial_bacc", "trial_macro_f1", "trial_weighted_f1",
  ]
  summary = pd.DataFrame({m: [df_loso[m].mean()] for m in metrics})
  print(summary.to_string(index=False, float_format=lambda v: "%.4f" % v))
  return df_loso


# =====================================================================
# 8. PROTOCOL 2: ROUND-ROBIN CROSS-SESSION EVALUATION
# =====================================================================
def run_round_robin_cross_session(stmae, x_pt, seq_idx_pt, packs):
  """Session-Wise Evaluation:
  Train Sess 1 -> Test Sess 2 & Test Sess 3 independently -> Average
  Train Sess 2 -> Test Sess 1 & Test Sess 3 independently -> Average
  Train Sess 3 -> Test Sess 1 & Test Sess 2 independently -> Average
  """
  labels, s_sub, s_ses, s_tri = packs[1:]
  groups = s_sub * 1000000 + s_ses * 10000 + s_tri
  sessions = [1, 2, 3]

  print("\n" + "=" * 92)
  print("PROTOCOL 2: SEED-IV SESSION-WISE ROUND-ROBIN CROSS-SESSION EVALUATION")
  print("=" * 92)

  session_pair_records = []
  anchor_summary_records = []

  for tr_ses in sessions:
    te_sessions = [s for s in sessions if s != tr_ses]
    print(
        f"\n[+] Training Anchor: SESSION {tr_ses} (Evaluating on Sessions "
        f"{te_sessions[0]} and {te_sessions[1]} independently)"
    )

    tr_rows = np.where(s_ses == tr_ses)[0]
    model = fit_model(
        copy.deepcopy(stmae), x_pt, seq_idx_pt, labels, groups, tr_rows
    )

    anchor_tests = []
    for te_ses in te_sessions:
      te_rows = np.where(s_ses == te_ses)[0]
      res = evaluate_test_rows(
          model, x_pt, seq_idx_pt, labels, s_sub, s_ses, s_tri, te_rows
      )
      res["task"] = f"Sess_{tr_ses}_to_Sess_{te_ses}"
      res["train_session"] = tr_ses
      res["test_session"] = te_ses
      session_pair_records.append(res)
      anchor_tests.append(res)

      print(
          f"  -> Task: Sess {tr_ses} -> Sess {te_ses:<2} | "
          f"WIN: acc={res['win_acc']:.4f} bacc={res['win_bacc']:.4f} macF1={res['win_macro_f1']:.4f} | "
          f"TRIAL: acc={res['trial_acc']:.4f} bacc={res['trial_bacc']:.4f} macF1={res['trial_macro_f1']:.4f} "
          f"({res['num_windows']}w / {res['num_trials']}t)"
      )

    df_anchor = pd.DataFrame(anchor_tests)
    anchor_avg = {
        "anchor_train_session": f"Train_Session_{tr_ses}_Average",
        "win_acc": df_anchor["win_acc"].mean(),
        "win_bacc": df_anchor["win_bacc"].mean(),
        "win_macro_f1": df_anchor["win_macro_f1"].mean(),
        "win_weighted_f1": df_anchor["win_weighted_f1"].mean(),
        "trial_acc": df_anchor["trial_acc"].mean(),
        "trial_bacc": df_anchor["trial_bacc"].mean(),
        "trial_macro_f1": df_anchor["trial_macro_f1"].mean(),
        "trial_weighted_f1": df_anchor["trial_weighted_f1"].mean(),
    }
    anchor_summary_records.append(anchor_avg)
    print(
        f"  >> [Anchor {tr_ses} Averaged] : "
        f"WIN Acc = {anchor_avg['win_acc']:.4f} | TRIAL Acc = {anchor_avg['trial_acc']:.4f} | "
        f"WIN F1 = {anchor_avg['win_macro_f1']:.4f} | TRIAL F1 = {anchor_avg['trial_macro_f1']:.4f}"
    )

  df_pairs = pd.DataFrame(session_pair_records)
  df_anchors = pd.DataFrame(anchor_summary_records)

  df_pairs.to_csv(
      os.path.join(OUTPUT_DIR, "results_seediv_cross_session_pairs.csv"),
      index=False,
  )
  df_anchors.to_csv(
      os.path.join(OUTPUT_DIR, "results_seediv_cross_session_anchors.csv"),
      index=False,
  )

  print("\n" + "-" * 92)
  print("SUMMARY: ANCHOR-AVERAGED SESSION RESULTS:")
  print("-" * 92)
  print(df_anchors.to_string(index=False, float_format=lambda v: "%.4f" % v))

  grand_avg = {
      "Metric": [
          "win_acc", "win_bacc", "win_macro_f1",
          "trial_acc", "trial_bacc", "trial_macro_f1",
      ],
      "Mean_Over_All_Sessions": [
          df_anchors["win_acc"].mean(),
          df_anchors["win_bacc"].mean(),
          df_anchors["win_macro_f1"].mean(),
          df_anchors["trial_acc"].mean(),
          df_anchors["trial_bacc"].mean(),
          df_anchors["trial_macro_f1"].mean(),
      ],
  }
  print("\n" + "-" * 92)
  print("GRAND AVERAGE OVER ALL 3 SESSION ANCHORS (SEED-IV):")
  print("-" * 92)
  print(
      pd.DataFrame(grand_avg).to_string(
          index=False, float_format=lambda v: "%.4f" % v
      )
  )

  return df_pairs, df_anchors


# =====================================================================
# 9. MAIN PIPELINE EXECUTION
# =====================================================================
def main():
  seed_everything(RANDOM_SEED)
  t0 = time.time()
  print(f"Device: {DEVICE}")
  print(f"Task: SEED-IV 4-Class Classification ({NUM_CLASSES} Emotions)")

  # 1. Load SEED-IV Data
  x_raw, y, subs, sess, tri = load_seed_iv_dataset()
  print(
      f"  [+] Data shape: {x_raw.shape} | Labels distribution: {np.bincount(y)}"
  )

  # 2. Normalize per Subject-Session
  x_norm = normalize_subject_session(x_raw, subs, sess)
  x_pt = torch.tensor(x_norm, dtype=torch.float32, device=DEVICE)

  # 3. Form Sequences (T=10, Stride=1)
  packs = build_sequence_index(
      y, subs, sess, tri, seq_len=WINDOW_LENGTH, stride=STRIDE
  )
  seq_idx_pt = torch.tensor(packs[0], dtype=torch.long, device=DEVICE)
  print(f"  [+] Formed {len(seq_idx_pt)} sequences of length {WINDOW_LENGTH}.")

  # 4. Global STMAE Pretraining
  stmae = pretrain_stmae_sequences(x_pt, seq_idx_pt, epochs=STMAE_EPOCHS)
  for p in stmae.parameters():
    p.requires_grad = False

  # 5. Run Leave-One-Subject-Out (LOSO) Protocol
  df_loso = run_loso_evaluation(stmae, x_pt, seq_idx_pt, packs)

  # 6. Run Round-Robin Session-Wise Cross-Session Protocol
  df_pairs, df_anchors = run_round_robin_cross_session(
      stmae, x_pt, seq_idx_pt, packs
  )

  print(
      f"\n[+] Finished SEED-IV Pipeline in {(time.time() - t0) / 60.0:.2f} minutes."
  )


if __name__ == "__main__":
  main()

Device: cuda
Task: SEED-IV 4-Class Classification (4 Emotions)
  [+] Ingesting SEED-IV features from: /kaggle/input/datasets/phhasian0710/seed-iv/eeg_feature_smooth
  [+] Successfully loaded 37575 SEED-IV frames.
  [+] Data shape: (37575, 62, 10) | Labels distribution: [10170 10245  9225  7935]
  [+] Formed 27855 sequences of length 10.
  [+] Starting Spatio-Temporal Masked Autoencoder Pretraining...
    -> STMAE Epoch 01 | Joint Recon MSE: 0.34668
    -> STMAE Epoch 05 | Joint Recon MSE: 0.13856
    -> STMAE Epoch 10 | Joint Recon MSE: 0.11900
    -> STMAE Epoch 15 | Joint Recon MSE: 0.10937
    -> STMAE Epoch 20 | Joint Recon MSE: 0.10403
    -> STMAE Epoch 25 | Joint Recon MSE: 0.09932
    -> STMAE Epoch 30 | Joint Recon MSE: 0.09698
    -> STMAE Epoch 35 | Joint Recon MSE: 0.09661

PROTOCOL 1: SEED-IV 4-CLASS LOSO CROSS-VALIDATION (15 SUBJECTS)
  -> Subject_01   | WIN: acc=0.5902 bacc=0.5909 macF1=0.5781 | TRIAL: acc=0.5694 bacc=0.5694 macF1=0.5631 (1857w / 72t)
  -> Subject_02   |